# N13 — Safety Car Probability: EDA & Labeling

The goal of this notebook is to build the labeled dataset for the safety car probability
model (N14). We work at the level of **race laps**: one row per (race, lap_number),
aggregating the observable state of the field at that point in the race.

For each lap we determine whether a Safety Car or Virtual Safety Car deployment
occurred within the next 5 laps — defining the binary `sc_within_5_laps` label.

The output is a Parquet file with race-state features and a binary SC label,
ready for Random Forest training in N14.

> **Dataset note:** unlike the overtake dataset (~28k pairs), the SC dataset is
> significantly smaller — ~70 races × ~55 laps ≈ ~4,000 rows. Class imbalance
> and validation strategy are designed accordingly.

**Data sources:**
- FastF1: TrackStatus, lap times, positions, weather, retirements (2023–2025)
- `data/processed/circuit_clustering/`: circuit cluster assignments

**Exports:**
- EDA plots → `notebooks/strategy/sc_probability/outputs/`
- Labeled dataset → `data/processed/sc_labeled/`


---

## Step 0 - Setup and Imports 

In [1]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import pathlib
import json
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS    = repo_root / "notebooks" / "strategy" / "sc_probability" / "outputs"
PROCESSED  = repo_root / "data" / "processed" / "sc_labeled"
CACHE      = repo_root / "data" / "cache" / "fastf1"
CLUSTERING = repo_root / "data" / "processed" / "circuit_clustering"

OUTPUTS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE))


In [2]:
print(f"Repo root  : {repo_root}")
print(f"Outputs    : {OUTPUTS}")
print(f"Processed  : {PROCESSED}")
print(f"Clustering : {CLUSTERING}")
print(f"FF1 cache  : {CACHE}")

Repo root  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Outputs    : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\strategy\sc_probability\outputs
Processed  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\sc_labeled
Clustering : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\circuit_clustering
FF1 cache  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\cache\fastf1


---

## Step 1 — Schedule & Circuit Clusters

Load the race calendar for 2023–2025 and attach the circuit cluster label to each
event. The cluster (0–3) encodes the circuit type (power, high-downforce, balanced,
street) and will be used as a categorical feature in N14.

**Circuit cluster source:** `data/processed/circuit_clustering/circuit_clusters_k4.parquet`
— produced in the circuit clustering notebook and reused across all strategy models.


In [3]:
# ── Step 1 — Load schedule + circuit clusters ─────────────────────────────────
SEASONS = [2023, 2024, 2025]

# ── Circuit cluster mapping ────────────────────────────────────────────────────
_clusters_df = pd.read_parquet(CLUSTERING / "circuit_clusters_k4.parquet")
cluster_map = dict(zip(_clusters_df["GP_Name"], _clusters_df["Cluster"]))

print(f"Circuit clusters loaded: {len(cluster_map)} circuits")
print(cluster_map)

# ── Race schedule (exclude testing, sprints count as separate events) ──────────
schedule_rows = []
for year in SEASONS:
    sched = fastf1.get_event_schedule(year, include_testing=False)
    races = sched[sched["EventFormat"].isin(["conventional", "sprint_shootout", "sprint"])]
    # Keep only rounds with a Race session
    races = races[races["Session5"] == "Race"].copy()
    races["year"] = year
    schedule_rows.append(races[["year", "RoundNumber", "EventName", "Location", "Country"]])

schedule = pd.concat(schedule_rows, ignore_index=True)

# ── Attach circuit cluster ─────────────────────────────────────────────────────
def resolve_cluster(location: str) -> int:
    for key, cluster in cluster_map.items():
        if key.lower() in location.lower() or location.lower() in key.lower():
            return cluster
    return -1   # unknown

schedule["circuit_cluster"] = schedule["Location"].apply(resolve_cluster)

unknown = schedule[schedule["circuit_cluster"] == -1]
print(f"\nSchedule loaded: {len(schedule)} races across {SEASONS}")
print(f"Unknown clusters: {len(unknown)}")
if len(unknown):
    print(unknown[["year", "EventName", "Location"]].to_string(index=False))

print(f"\nCluster distribution:\n{schedule['circuit_cluster'].value_counts().sort_index()}")


Circuit clusters loaded: 25 circuits
{'Austin': 1, 'Baku': 1, 'Barcelona': 3, 'Budapest': 3, 'Imola': 1, 'Jeddah': 1, 'Las Vegas': 0, 'Lusail': 0, 'Marina Bay': 1, 'Melbourne': 2, 'Mexico City': 3, 'Miami': 1, 'Miami Gardens': 1, 'Monaco': 3, 'Montréal': 2, 'Monza': 1, 'Sakhir': 0, 'Shanghai': 0, 'Silverstone': 0, 'Spa-Francorchamps': 0, 'Spielberg': 3, 'Suzuka': 0, 'São Paulo': 2, 'Yas Island': 1, 'Zandvoort': 2}

Schedule loaded: 58 races across [2023, 2024, 2025]
Unknown clusters: 0

Cluster distribution:
circuit_cluster
0    15
1    19
2    10
3    14
Name: count, dtype: int64
